<a href="https://colab.research.google.com/github/kokami236/osiro1/blob/main/endewakeru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install open3d


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 144.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 82.0 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


点群の数の確認


In [3]:
import open3d as o3d
import numpy as np

def count_points_with_open3d(file_path):
    # .plyファイルを読み込む
    print(f"ファイルを読み込んでいます: {file_path}")
    pcd = o3d.io.read_point_cloud(file_path)

    # 点群データが存在するか確認
    if pcd.is_empty():
        print("点群データが空、もしくはファイルの読み込みに失敗しました。")
        return

    # 点群の数を取得 (numpy配列に変換して形状を確認、またはlen()を使用)
    point_count = len(pcd.points)

    print(f"--- 結果 ---")
    print(f"点群の数: {point_count}")

    # 必要であればnumpy配列として詳細データを取得可能
    # points_np = np.asarray(pcd.points)
    # print(points_np.shape)

# ファイルパスを指定して実行
file_path = "/content/drive/MyDrive/ジオラマ/201903082.ply"  # ここに読み込みたいファイル名を指定
count_points_with_open3d(file_path)

ファイルを読み込んでいます: /content/drive/MyDrive/ジオラマ/201903082.ply
--- 結果 ---
点群の数: 140304411


ダウンサンプリングのみ


In [5]:
import open3d as o3d
import os

# =============================
# ユーザー設定
# =============================
input_file  = "/content/drive/MyDrive/ジオラマ/201903082.ply"
output_file = "/content/drive/MyDrive/ジオラマ/kakegawa2_downsampled.ply"

voxel_size = 0.01  # 例: 0.01 = 1cm

# =============================
# 実行
# =============================
if not os.path.exists(input_file):
    raise FileNotFoundError(f"入力ファイルが見つかりません: {input_file}")

print("読み込み中...")
pcd = o3d.io.read_point_cloud(input_file)

if not pcd.has_points():
    raise ValueError("点が含まれていません")

print(f"元の点数: {len(pcd.points)}")
print(f"元データ: colors={pcd.has_colors()}, normals={pcd.has_normals()}")

print(f"voxel_size={voxel_size} でダウンサンプリング中...")
pcd_ds = pcd.voxel_down_sample(voxel_size=voxel_size)

print(f"ダウンサンプリング後の点数: {len(pcd_ds.points)}")
print(f"後データ: colors={pcd_ds.has_colors()}, normals={pcd_ds.has_normals()}")

# ★色情報チェック（点数と色配列の整合）
if pcd_ds.has_colors():
    if len(pcd_ds.colors) != len(pcd_ds.points):
        print("⚠️ 色配列の長さが点数と一致していません（異常）")
    else:
        print("✅ 色は保持されています（点数とcolors数が一致）")
else:
    print("⚠️ そもそも入力に色情報が無い or 読み込み時に色として解釈されていません")

print("保存中...")
# write_ascii=False でバイナリ保存（軽い・速い）。ASCIIにしたければ True
ok = o3d.io.write_point_cloud(output_file, pcd_ds, write_ascii=False)
print("保存完了:" if ok else "保存失敗:", output_file)


読み込み中...
元の点数: 140304411
元データ: colors=True, normals=False
voxel_size=0.01 でダウンサンプリング中...
ダウンサンプリング後の点数: 35112759
後データ: colors=True, normals=False
✅ 色は保持されています（点数とcolors数が一致）
保存中...
保存完了: /content/drive/MyDrive/ジオラマ/kakegawa2_downsampled.ply


縮尺関係表示

In [6]:
import open3d as o3d
import numpy as np

path = "/content/drive/MyDrive/ジオラマ/kakegawa2_downsampled.ply"
pcd = o3d.io.read_point_cloud(path)

pts = np.asarray(pcd.points)
mn = pts.min(axis=0)
mx = pts.max(axis=0)
extent = mx - mn

print("min:", mn)
print("max:", mx)
print("extent (X,Y,Z):", extent)
print("diagonal length:", np.linalg.norm(extent))

center = pcd.get_center()
print("center:", center)


min: [-4.45078890e+04 -1.35793794e+05  4.55610008e+01]
max: [-4.44631560e+04 -1.35751666e+05  7.35996552e+01]
extent (X,Y,Z): [44.73300171 42.12800217 28.03865433]
diagonal length: 67.54240257015246
center: [-4.44899141e+04 -1.35769914e+05  5.82778851e+01]


In [7]:
import open3d as o3d
import numpy as np

input_path  = "/content/drive/MyDrive/ジオラマ/kakegawa2_downsampled.ply"
output_path = "/content/drive/MyDrive/ジオラマ/kakegawa2_norm_diag1p7.ply"

target_diag = 1.7  # ここを「揃えたいサイズ」にする（熊本城運用に合わせる）

pcd = o3d.io.read_point_cloud(input_path)
pts = np.asarray(pcd.points)

mn = pts.min(axis=0)
mx = pts.max(axis=0)
extent = mx - mn
diag = np.linalg.norm(extent)
center = (mn + mx) / 2

print("before diag:", diag)
print("before center:", center)

# 1) 中心を原点へ移動（平行移動）
pts0 = pts - center

# 2) スケール調整（対角をtarget_diagへ）
scale = target_diag / diag
pts1 = pts0 * scale

pcd2 = o3d.geometry.PointCloud()
pcd2.points = o3d.utility.Vector3dVector(pts1)

# 色があるならコピー（点数は変わらないので対応OK）
if pcd.has_colors():
    pcd2.colors = pcd.colors

print("scale factor:", scale)

# 保存
o3d.io.write_point_cloud(output_path, pcd2, write_ascii=False)
print("saved:", output_path)


before diag: 67.54240257015246
before center: [-4.44855225e+04 -1.35772730e+05  5.95803280e+01]
scale factor: 0.025169374131077237
saved: /content/drive/MyDrive/ジオラマ/kakegawa2_norm_diag1p7.ply


正規化を上でしたので半径1.5まで切り分ける


In [11]:
import numpy as np
import open3d as o3d

pcd = o3d.io.read_point_cloud(
    "/content/drive/MyDrive/ジオラマ/kakegawa2_norm_diag1p7.ply"
)
pts = np.asarray(pcd.points)
center = pcd.get_center()

dists = np.linalg.norm(pts - center, axis=1)
print("max distance:", dists.max())
print("mean distance:", dists.mean())


max distance: 0.7928401977572943
mean distance: 0.21569774184676496


In [15]:
import open3d as o3d
import numpy as np
import json
import os

# --- 入力: 1.7で切ったPLY ---
input_ply = "/content/drive/MyDrive/ジオラマ/kakegawa2_norm_diag1p7.ply"
# --- 出力: 1.5でさらに切る ---
output_ply = "/content/drive/MyDrive/ジオラマ/kakegawa_seikai3.ply"

# ★ 1.7を作ったときの中心を保存してあるならそれを使うのがベスト
# 例: center_json に {"center":[x,y,z]} が入ってる想定
center_json = "/content/drive/MyDrive/ジオラマ/kakegawa_center.json"

radius = 0.4

assert os.path.exists(input_ply), f"Not found: {input_ply}"

pcd = o3d.io.read_point_cloud(input_ply)
pts = np.asarray(pcd.points)
print("Loaded:", len(pts), "Has colors:", pcd.has_colors())

if os.path.exists(center_json):
    with open(center_json, "r") as f:
        center = np.array(json.load(f)["center"], dtype=np.float64)
    print("Use saved center:", center)
else:
    # 保存してない場合は、妥協してここで重心を使う（中心が少しズレる可能性あり）
    center = np.array(pcd.get_center(), dtype=np.float64)
    print("Use current center (fallback):", center)

dist = np.linalg.norm(pts - center, axis=1)
idx = np.where(dist <= radius)[0]
out = pcd.select_by_index(idx)

print("Remaining:", len(out.points), "Reduction:", 100*(1-len(out.points)/len(pcd.points)), "%")
print("Out has colors:", out.has_colors())

o3d.io.write_point_cloud(output_ply, out, write_ascii=False)
print("Saved:", output_ply)


Loaded: 35112759 Has colors: True
Use current center (fallback): [-0.11053316  0.07086904 -0.03278167]
Remaining: 32642691 Reduction: 7.034673635301624 %
Out has colors: True
Saved: /content/drive/MyDrive/ジオラマ/kakegawa_seikai3.ply


In [16]:
import open3d as o3d
import numpy as np
import copy

# 入力：radius=0.4で切った点群
input_path  = "/content/drive/MyDrive/ジオラマ/kakegawa_seikai3.ply"
output_path = "/content/drive/MyDrive/ジオラマ/kakegawa_seikai4.ply"

target_diameter = 1.7  # SeedFormer想定サイズ

pcd = o3d.io.read_point_cloud(input_path)
pts = np.asarray(pcd.points)

# 1. 中心を原点へ
center = pcd.get_center()
pts0 = pts - center

# 2. 現在の最大半径を計測
dists = np.linalg.norm(pts0, axis=1)
current_radius = dists.max()
current_diameter = current_radius * 2

print("current radius:", current_radius)
print("current diameter:", current_diameter)

# 3. スケール係数
scale = target_diameter / current_diameter
print("scale factor:", scale)

# 4. スケール適用
pts1 = pts0 * scale

pcd2 = o3d.geometry.PointCloud()
pcd2.points = o3d.utility.Vector3dVector(pts1)

# 色があれば保持
if pcd.has_colors():
    pcd2.colors = pcd.colors

# 確認
dists_after = np.linalg.norm(np.asarray(pcd2.points), axis=1)
print("after max radius:", dists_after.max())
print("after diameter:", dists_after.max() * 2)

# 保存
o3d.io.write_point_cloud(output_path, pcd2, write_ascii=False)
print("Saved:", output_path)


current radius: 0.4220005537537666
current diameter: 0.8440011075075332
scale factor: 2.0142153663996543
after max radius: 0.85
after diameter: 1.7
Saved: /content/drive/MyDrive/ジオラマ/kakegawa_seikai4.ply


比較

In [19]:
import open3d as o3d
import numpy as np

def analyze_pointcloud(path, name):
    pcd = o3d.io.read_point_cloud(path)
    pts = np.asarray(pcd.points)

    center = pcd.get_center()
    mn = pts.min(axis=0)
    mx = pts.max(axis=0)
    extent = mx - mn
    diag = float(np.linalg.norm(extent))

    dists = np.linalg.norm(pts - center, axis=1)
    max_radius = float(dists.max())
    mean_radius = float(dists.mean())

    print(f"\n=== {name} ===")
    print("points:", len(pts))
    print("extent (X,Y,Z):", extent)
    print("diagonal:", diag)
    print("max radius:", max_radius)
    print("mean radius:", mean_radius)

    return {
        "extent": extent,
        "diag": diag,
        "max_radius": max_radius,
        "mean_radius": mean_radius
    }

# =========================
# 入力（ここを書き換える）
# =========================
kumamoto_path = "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply"
kakegawa_path = "/content/drive/MyDrive/ジオラマ/kakegawa_scaled_to_kumamoto.ply"

kumamoto = analyze_pointcloud(kumamoto_path, "Kumamoto Castle")
kakegawa = analyze_pointcloud(kakegawa_path, "Kakegawa Castle")

# =========================
# スケール比較
# =========================
print("\n=== Scale comparison ===")

diag_ratio = kakegawa["diag"] / kumamoto["diag"]
radius_ratio = kakegawa["max_radius"] / kumamoto["max_radius"]

print("Diagonal ratio (kakegawa / kumamoto):", diag_ratio)
print("Max radius ratio:", radius_ratio)

# 判定（SeedFormer用）
tol = 0.05  # 5%誤差までOK
if abs(diag_ratio - 1.0) < tol and abs(radius_ratio - 1.0) < tol:
    print("✅ スケールは一致（SeedFormerにそのまま投入OK）")
else:
    print("⚠️ スケールが不一致 → 正規化が必要")



=== Kumamoto Castle ===
points: 970651
extent (X,Y,Z): [4.99355674 3.29972279 3.44459391]
diagonal: 6.90572274061408
max radius: 3.002451554114631
mean radius: 1.037487022845833

=== Kakegawa Castle ===
points: 32642691
extent (X,Y,Z): [5.51318744 5.3166265  4.34089955]
diagonal: 8.80370160836456
max radius: 3.002451554114665
mean radius: 1.3779980780413938

=== Scale comparison ===
Diagonal ratio (kakegawa / kumamoto): 1.274841452378047
Max radius ratio: 1.0000000000000113
⚠️ スケールが不一致 → 正規化が必要


In [18]:
import open3d as o3d
import numpy as np

# =========================
# 入力 / 出力
# =========================
kakegawa_input  = "/content/drive/MyDrive/ジオラマ/kakegawa_seikai4.ply"
kakegawa_output = "/content/drive/MyDrive/ジオラマ/kakegawa_scaled_to_kumamoto.ply"

# 熊本城の基準値（学習データから固定）
kumamoto_max_radius = 3.002451554114631  # ←あなたのログから

# =========================
# 読み込み
# =========================
pcd = o3d.io.read_point_cloud(kakegawa_input)
pts = np.asarray(pcd.points)

print("Loaded points:", len(pts))
print("Has colors:", pcd.has_colors())

# =========================
# 1. 中心合わせ
# =========================
center = pcd.get_center()
pts0 = pts - center

# =========================
# 2. 掛川城の現在半径
# =========================
dists = np.linalg.norm(pts0, axis=1)
kakegawa_radius = dists.max()

print("Kakegawa max radius (before):", kakegawa_radius)

# =========================
# 3. スケール計算
# =========================
scale = kumamoto_max_radius / kakegawa_radius
print("Scale factor:", scale)

# =========================
# 4. スケール適用
# =========================
pts1 = pts0 * scale

pcd_out = o3d.geometry.PointCloud()
pcd_out.points = o3d.utility.Vector3dVector(pts1)

# 色があれば保持
if pcd.has_colors():
    pcd_out.colors = pcd.colors

# =========================
# 5. 確認
# =========================
dists_after = np.linalg.norm(np.asarray(pcd_out.points), axis=1)
print("After max radius:", dists_after.max())
print("Expected radius:", kumamoto_max_radius)

# =========================
# 6. 保存
# =========================
o3d.io.write_point_cloud(kakegawa_output, pcd_out, write_ascii=False)
print("Saved:", kakegawa_output)


Loaded points: 32642691
Has colors: True
Kakegawa max radius (before): 0.8499999999999559
Scale factor: 3.532295946017396
After max radius: 3.002451554114631
Expected radius: 3.002451554114631
Saved: /content/drive/MyDrive/ジオラマ/kakegawa_scaled_to_kumamoto.ply


ただダウンサンプリング

In [20]:
import open3d as o3d
import numpy as np
import os

# =========================
# 入出力
# =========================
input_path  = "/content/drive/MyDrive/ジオラマ/kakegawa_scaled_to_kumamoto.ply"
output_path = "/content/drive/MyDrive/ジオラマ/kakegawa_scaled_to_kumamoto_downsampled.ply"

voxel_size = 0.005  # ← 好きに調整（例: 0.005=5mm, 0.01=1cm）

# =========================
# 実行
# =========================
assert os.path.exists(input_path), f"Not found: {input_path}"

pcd = o3d.io.read_point_cloud(input_path)
print("Loaded points:", len(pcd.points))
print("Has colors:", pcd.has_colors())

pcd_ds = pcd.voxel_down_sample(voxel_size=voxel_size)

print("After downsample:", len(pcd_ds.points))
print("Downsampled has colors:", pcd_ds.has_colors())

# 色が落ちてないか確認
if pcd.has_colors() and not pcd_ds.has_colors():
    print("⚠️ WARNING: colors dropped (should not happen normally)")

o3d.io.write_point_cloud(output_path, pcd_ds, write_ascii=False)
print("Saved:", output_path)


Loaded points: 32642691
Has colors: True
After downsample: 6975229
Downsampled has colors: True
Saved: /content/drive/MyDrive/ジオラマ/kakegawa_scaled_to_kumamoto_downsampled.ply


ここから先前のデータセットで使ってただけ

上ではただ円で切り分けるだけ、ダウンサンプリングなし
したコードをそのあとに適応する


In [ ]:
import open3d as o3d
import numpy as np
import os

# ==========================================
# --- ユーザー設定エリア ---
# ==========================================

# 1. 入力ファイル
input_file = "/content/drive/MyDrive/ジオラマ/201903082.ply"

# 2. 保存ファイル
output_file = "/content/drive/MyDrive/ジオラマ/kakegawa3.ply"

# 3. 残したい半径 (メートル単位)
# ここで設定した半径より外側をカットします
radius = 1.7

# ==========================================
# --- 処理実行エリア ---
# ==========================================

if not os.path.exists(input_file):
    print(f"エラー: 入力ファイルが見つかりません: {input_file}")
else:
    # 1. ファイル読み込み
    print("ファイルを読み込んでいます...")
    pcd = o3d.io.read_point_cloud(input_file)

    if not pcd.has_points():
        print("エラー: 点が含まれていません。")
    else:
        original_count = len(pcd.points)
        print(f"元の点の数: {original_count}")

        # ------------------------------------------
        # 2. 重心の計算 (元の高密度なデータから計算)
        # ------------------------------------------
        center = pcd.get_center()
        print(f"データの重心: {center}")

        # ------------------------------------------
        # 3. 半径による範囲カット
        # ------------------------------------------
        print(f"半径 {radius}m 以内の点を抽出中...")

        # 各点と重心との距離を計算
        points = np.asarray(pcd.points)
        distances = np.linalg.norm(points - center, axis=1)

        # 指定半径(radius)以内の点だけを選ぶ
        indices = np.where(distances <= radius)[0]

        # 抽出を実行
        final_pcd = pcd.select_by_index(indices)

        # ------------------------------------------
        # 4. 結果の保存
        # ------------------------------------------
        final_count = len(final_pcd.points)

        # 削除された点の割合を計算
        if original_count > 0:
            reduction_rate = (1 - final_count / original_count) * 100
        else:
            reduction_rate = 0

        print(f"最終的に残った点の数: {final_count}")
        print(f"カットされた割合: {reduction_rate:.2f}%")

        o3d.io.write_point_cloud(output_file, final_pcd)
        print(f"保存完了: {output_file}")

ファイルを読み込んでいます...
元の点の数: 1073984
データの重心: [-0.18435264 -0.60698334 -0.20332932]
半径 1.7m 以内の点を抽出中...
最終的に残った点の数: 957301
カットされた割合: 10.86%
保存完了: /content/drive/MyDrive/ジオラマ/小倉城out2.ply


In [4]:
import open3d as o3d
import numpy as np
import os

# ==========================================
# --- ユーザー設定エリア ---
# ==========================================

# 1. 入力ファイル
input_file = "/content/drive/MyDrive/ジオラマ/201903082.ply"

# 2. 保存ファイル
output_file = "/content/drive/MyDrive/ジオラマ/kakegawa2.ply"

# 3. ボクセルサイズ (軽量化の度合い)
voxel_size = 0.01  # 例: 0.01 = 1cm間隔

# 4. 残したい半径 (メートル単位)
# 軽量化した後の重心から、この半径より外側をカットします
radius = 1.9

# ==========================================
# --- 処理実行エリア ---
# ==========================================

if not os.path.exists(input_file):
    print(f"エラー: 入力ファイルが見つかりません: {input_file}")
else:
    # 1. ファイル読み込み
    print("ファイルを読み込んでいます...")
    pcd = o3d.io.read_point_cloud(input_file)

    if not pcd.has_points():
        print("エラー: 点が含まれていません。")
    else:
        print(f"元の点の数: {len(pcd.points)}")

        # ------------------------------------------
        # 2. ボクセルダウンサンプリング（軽量化）
        # ------------------------------------------
        print(f"ボクセルサイズ {voxel_size} でダウンサンプリング中...")
        downsampled_pcd = pcd.voxel_down_sample(voxel_size=voxel_size)

        # 軽くなった時点での点数を表示
        temp_count = len(downsampled_pcd.points)
        print(f"ダウンサンプリング後の点の数: {temp_count}")

        if temp_count == 0:
            print("警告: 点がなくなりました。voxel_sizeを小さくしてください。")
        else:
            # ------------------------------------------
            # 3. 重心の再計算 & ノイズ削除
            # ------------------------------------------

            # ★ポイント: 「軽量化された点群(downsampled_pcd)」の重心を計算します
            center = downsampled_pcd.get_center()
            print(f"軽量化後のデータから計算した重心: {center}")

            # 各点と重心との距離を計算
            points = np.asarray(downsampled_pcd.points)
            distances = np.linalg.norm(points - center, axis=1)

            # 指定半径(radius)以内の点だけを選ぶ
            indices = np.where(distances <= radius)[0]
            final_pcd = downsampled_pcd.select_by_index(indices)

            # ------------------------------------------
            # 4. 結果の保存
            # ------------------------------------------
            final_count = len(final_pcd.points)
            reduction_rate = (1 - final_count / len(pcd.points)) * 100

            print(f"最終的に残った点の数: {final_count}")
            print(f"トータルの削減率: {reduction_rate:.2f}%")

            o3d.io.write_point_cloud(output_file, final_pcd)
            print(f"保存完了: {output_file}")

ファイルを読み込んでいます...
元の点の数: 140304411
ボクセルサイズ 0.01 でダウンサンプリング中...
ダウンサンプリング後の点の数: 35112759
軽量化後のデータから計算した重心: [-4.44899141e+04 -1.35769914e+05  5.82778851e+01]
最終的に残った点の数: 318089
トータルの削減率: 99.77%
保存完了: /content/drive/MyDrive/ジオラマ/kakegawa2.ply


In [ ]:
import open3d as o3d
import numpy as np
import os

# ==========================================
# --- 設定エリア（ステップ3） ---
# ==========================================

# 1. 入力ファイル
# 前のステップ（軽量化）で保存したファイルを指定します
input_file = "/content/drive/MyDrive/ジオラマ/小倉城out2.ply"

# 2. 最終的に残したい半径
# 軽量化によって重心位置がわずかに変わっている可能性があるため、
# ここで設定した半径で再度きれいに切り抜きます。
radius = 2.22

# 3. 保存するファイル名（完成データ）
output_file = "/content/drive/MyDrive/ジオラマ/小倉城_最終完成.ply"

# ==========================================
# --- 処理実行 ---
# ==========================================

if not os.path.exists(input_file):
    print(f"エラー: 前のステップの出力ファイルが見つかりません: {input_file}")
    print("一つ前のコードセル（軽量化処理）が正しく実行されているか確認してください。")
else:
    # ファイル読み込み
    pcd = o3d.io.read_point_cloud(input_file)

    if not pcd.has_points():
        print("エラー: 点群データが空です。")
    else:
        print(f"軽量化済みデータの点数: {len(pcd.points)}")

        # --- 重心の再計算 ---
        # 軽くなったデータに基づいて、改めて重心を求めます
        center = pcd.get_center()
        print(f"再計算された重心座標: {center}")

        # --- 距離によるフィルタリング（円形/球形カット） ---
        points = np.asarray(pcd.points)
        distances = np.linalg.norm(points - center, axis=1)

        # 半径以内のインデックスを取得
        indices = np.where(distances <= radius)[0]

        # 抽出実行
        final_pcd = pcd.select_by_index(indices)

        # --- 結果表示と保存 ---
        final_count = len(final_pcd.points)
        removed_count = len(pcd.points) - final_count

        print(f"半径 {radius}m 以内に残った点数: {final_count}")
        print(f"削除されたノイズ（外側の点）の数: {removed_count}")

        o3d.io.write_point_cloud(output_file, final_pcd)
        print(f"全ての処理が完了しました。保存先: {output_file}")

In [ ]:
import open3d as o3d
import numpy as np
import os

# --- ユーザーが設定する項目 ---

# 1. 入力する点群ファイルのパス
input_file = "/content/drive/MyDrive/熊本城外きれい1.ply"

# 2. 中心の点から残したい半径 (単位は点群の座標系に依存します)
# この値を大きくすると、より多くの点が残ります。
radius = 3.0

# 3. 保存するファイル名
output_file = "/content/drive/MyDrive/centered_point_cloud.ply"

# --------------------------


# ファイルの存在を確認
if not os.path.exists(input_file):
    print(f"エラー: 入力ファイルが見つかりません: {input_file}")
else:
    # 点群データを読み込む
    pcd = o3d.io.read_point_cloud(input_file)

    if not pcd.has_points():
        print("エラー: 点群の読み込みに失敗したか、点が含まれていません。")
    else:
        print(f"処理前の点の数: {len(pcd.points)}")

        # 1. 点群の重心（中心）を計算
        center = pcd.get_center()
        print(f"計算された中心座標: {center}")

        # 2. 各点が中心からどれだけ離れているか計算
        points = np.asarray(pcd.points)

        # NumPyを使って全点の中心からの距離を高速に計算
        distances = np.linalg.norm(points - center, axis=1)

        # 3. 指定した半径の内側にある点のインデックスを取得
        indices = np.where(distances <= radius)[0]

        # 4. 半径内の点群だけを抽出して新しい点群オブジェクトを作成
        centered_pcd = pcd.select_by_index(indices)

        print(f"半径 {radius} m 内の点の数: {len(centered_pcd.points)}")

        # 5. 結果をファイルに書き出す
        o3d.io.write_point_cloud(output_file, centered_pcd)
        print(f"処理後のファイルを {output_file} に保存しました。")

処理前の点の数: 1093108
計算された中心座標: [ 0.59900511 -0.193925    0.99813606]
半径 3.0 m 内の点の数: 1029450
処理後のファイルを /content/drive/MyDrive/centered_point_cloud.ply に保存しました。
